In [2]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print(
    combined_df.groupby(
        ["Patient_ID","label"]
    ).size()
)

Patient_ID  label    
1855        before_HI     29
            during_HI     32
26398       before_HI     30
            during_HI     30
26502       before_HI     30
            during_HI     58
27245       before_HI     30
            during_HI     59
30071       before_HI     30
            during_HI     49
30851       before_HI     33
            during_HI     20
57091       before_HI     30
            during_HI     15
59285       before_HI     30
            during_HI     34
638         before_HI     24
            during_HI     36
70447       before_HI     29
            during_HI     43
8452        before_HI     30
            during_HI     59
98182       during_HI    107
dtype: int64


98182 has no preHI data.

In [ ]:
combined_df = combined_df[
    combined_df["Patient_ID"] != "98182"
].copy()

print(
    combined_df["Patient_ID"].nunique()
)

11


In [ ]:
combined_df.groupby(
    ["Patient_ID","label"]
).size()

Patient_ID  label    
1855        before_HI    29
            during_HI    32
26398       before_HI    30
            during_HI    30
26502       before_HI    30
            during_HI    58
27245       before_HI    30
            during_HI    59
30071       before_HI    30
            during_HI    49
30851       before_HI    33
            during_HI    20
57091       before_HI    30
            during_HI    15
59285       before_HI    30
            during_HI    34
638         before_HI    24
            during_HI    36
70447       before_HI    29
            during_HI    43
8452        before_HI    30
            during_HI    59
dtype: int64

Analysis A -- Nested LOSO

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    "/content/drive/MyDrive/2min_12_Patients_35Features.csv"
)

# ============================================================
# CREATE PATIENT ID
# ============================================================

df["Patient_ID"] = (
    df["File_Name"]
    .astype(str)
    .str.extract(r'^(\d+)')
)

# ============================================================
# REMOVE INCOMPLETE PATIENT
# ============================================================

df = df[df["Patient_ID"] != "98182"].copy()

print("Patients:", df["Patient_ID"].nunique())

# ============================================================
# ENCODE LABELS
# ============================================================

df["y"] = df["label"].map({
    "before_HI": 0,
    "during_HI": 1
})

# ============================================================
# FEATURE LIST
# ============================================================

exclude_cols = [
    "Sl. No:",
    "Segment_Number",
    "File_Name",
    "Patient_ID",
    "label",
    "y"
]

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
]

print("Features:", len(feature_cols))

# ============================================================
# LOSO SETUP
# ============================================================

logo = LeaveOneGroupOut()

groups = df["Patient_ID"]

all_probs = []
all_true = []

feature_frequency = {}

# ============================================================
# LOSO LOOP
# ============================================================

fold = 1

for train_idx, test_idx in logo.split(
        df,
        df["y"],
        groups):

    print(f"\nFold {fold}")

    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    # --------------------------------------------------------
    # PATIENT-LEVEL AGGREGATION
    # --------------------------------------------------------

    patient_train = (
        train_df
        .groupby(["Patient_ID", "label"])[feature_cols]
        .median()
        .reset_index()
    )

    # --------------------------------------------------------
    # EFFECT SIZE FEATURE RANKING
    # --------------------------------------------------------

    scores = {}

    for feature in feature_cols:

        before = patient_train[
            patient_train["label"] == "before_HI"
        ][feature]

        during = patient_train[
            patient_train["label"] == "during_HI"
        ][feature]

        score = abs(
            np.median(during.values)
            -
            np.median(before.values)
        )

        scores[feature] = score

    # --------------------------------------------------------
    # TOP 10 FEATURES
    # --------------------------------------------------------

    selected = sorted(
        scores,
        key=scores.get,
        reverse=True
    )[:10]

    # --------------------------------------------------------
    # CORRELATION PRUNING
    # --------------------------------------------------------

    corr = (
        train_df[selected]
        .corr()
        .abs()
    )

    upper = corr.where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )

    remove = [
        col
        for col in upper.columns
        if any(upper[col] > 0.90)
    ]

    selected = [
        f for f in selected
        if f not in remove
    ]

    print("Selected:", selected)

    # --------------------------------------------------------
    # FEATURE STABILITY
    # --------------------------------------------------------

    for f in selected:

        feature_frequency[f] = (
            feature_frequency.get(f, 0)
            + 1
        )

    # --------------------------------------------------------
    # SCALING
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train_df[selected]
    )

    X_test = scaler.transform(
        test_df[selected]
    )

    y_train = train_df["y"]
    y_test = test_df["y"]

    # --------------------------------------------------------
    # CLASSIFIER
    # --------------------------------------------------------

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    probs = model.predict_proba(
        X_test
    )[:, 1]

    all_probs.extend(probs)
    all_true.extend(y_test)

    fold += 1

# ============================================================
# FINAL LOSO AUC
# ============================================================

auc = roc_auc_score(
    all_true,
    all_probs
)

print("\n" + "="*60)
print("REVIEWER-SAFE NESTED LOSO RESULT")
print("="*60)
print(f"AUC = {auc:.4f}")

# ============================================================
# FEATURE STABILITY
# ============================================================

stability_df = pd.DataFrame(
    feature_frequency.items(),
    columns=[
        "Feature",
        "Selection_Count"
    ]
)

stability_df["Selection_%"] = (
    stability_df["Selection_Count"]
    /
    df["Patient_ID"].nunique()
    * 100
)

stability_df = stability_df.sort_values(
    "Selection_%",
    ascending=False
)

print("\n" + "="*60)
print("FEATURE STABILITY")
print("="*60)

print(stability_df)

# ============================================================
# SAVE FEATURE STABILITY
# ============================================================

stability_df.to_csv(
    "/content/drive/MyDrive/Feature_Stability_11Patients.csv",
    index=False
)

print("\nFeature stability saved.")

Patients: 11
Features: 35

Fold 1
Selected: ['fd_variance', 'fd_kurtosis', 'numberOfZeroCrossing', 'lempelZivComplexity', 'fd_mean', 'fd_median', 'kurtosis']

Fold 2
Selected: ['fd_kurtosis', 'fd_variance', 'numberOfZeroCrossing', 'lempelZivComplexity', 'kurtosis', 'fd_median', 'fd_mean']

Fold 3
Selected: ['fd_kurtosis', 'fd_variance', 'numberOfZeroCrossing', 'fd_median', 'fd_mean', 'kurtosis', 'katzFd']

Fold 4
Selected: ['fd_kurtosis', 'fd_variance', 'numberOfZeroCrossing', 'fd_maximum', 'kurtosis', 'fd_median', 'fd_mean', 'skewness']

Fold 5
Selected: ['fd_kurtosis', 'fd_variance', 'numberOfZeroCrossing', 'lempelZivComplexity', 'kurtosis', 'skewness', 'fd_mean']

Fold 6
Selected: ['fd_kurtosis', 'fd_variance', 'numberOfZeroCrossing', 'fd_mean', 'fd_median', 'kurtosis', 'skewness']

Fold 7
Selected: ['fd_variance', 'fd_kurtosis', 'numberOfZeroCrossing', 'lempelZivComplexity', 'fd_mean', 'fd_median', 'kurtosis']

Fold 8
Selected: ['fd_kurtosis', 'fd_variance', 'numberOfZeroCrossing',

Analysis B- patient normalization


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    "/content/drive/MyDrive/2min_12_Patients_35Features.csv"
)

# ============================================================
# PATIENT ID
# ============================================================

df["Patient_ID"] = (
    df["File_Name"]
    .astype(str)
    .str.extract(r'^(\d+)')
)

# Remove incomplete patient
df = df[df["Patient_ID"] != "98182"].copy()

# Labels
df["y"] = df["label"].map({
    "before_HI": 0,
    "during_HI": 1
})

# ============================================================
# FEATURES
# ============================================================

exclude_cols = [
    "Sl. No:",
    "Segment_Number",
    "File_Name",
    "Patient_ID",
    "label",
    "y"
]

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
]

# ============================================================
# LOSO
# ============================================================

logo = LeaveOneGroupOut()

groups = df["Patient_ID"]

all_probs = []
all_true = []

feature_frequency = {}

# patient-level results
patient_probs = []
patient_labels = []

# ============================================================
# LOSO LOOP
# ============================================================

fold = 1

for train_idx, test_idx in logo.split(
        df,
        df["y"],
        groups):

    print(f"\nFold {fold}")

    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    # --------------------------------------------------------
    # FEATURE MATRIX
    # --------------------------------------------------------

    X_train_full = train_df[feature_cols]
    y_train = train_df["y"]

    # --------------------------------------------------------
    # SCALING
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train_full
    )

    # --------------------------------------------------------
    # MUTUAL INFORMATION
    # --------------------------------------------------------

    mi = mutual_info_classif(
        X_train_scaled,
        y_train,
        random_state=42
    )

    mi_df = pd.DataFrame({
        "Feature": feature_cols,
        "MI": mi
    })

    selected = (
        mi_df
        .sort_values("MI", ascending=False)
        .head(10)
        ["Feature"]
        .tolist()
    )

    # --------------------------------------------------------
    # CORRELATION PRUNING
    # --------------------------------------------------------

    corr = (
        train_df[selected]
        .corr()
        .abs()
    )

    upper = corr.where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )

    remove = [
        col
        for col in upper.columns
        if any(upper[col] > 0.90)
    ]

    selected = [
        f for f in selected
        if f not in remove
    ]

    print("Selected:", selected)

    # --------------------------------------------------------
    # FEATURE STABILITY
    # --------------------------------------------------------

    for f in selected:

        feature_frequency[f] = (
            feature_frequency.get(f, 0)
            + 1
        )

    # --------------------------------------------------------
    # TRAIN/TEST DATA
    # --------------------------------------------------------

    X_train = scaler.fit_transform(
        train_df[selected]
    )

    X_test = scaler.transform(
        test_df[selected]
    )

    y_train = train_df["y"]
    y_test = test_df["y"]

    # --------------------------------------------------------
    # RANDOM FOREST
    # --------------------------------------------------------

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    probs = model.predict_proba(
        X_test
    )[:, 1]

    # Segment-level
    all_probs.extend(probs)
    all_true.extend(y_test)

    # --------------------------------------------------------
    # PATIENT-LEVEL AGGREGATION
    # --------------------------------------------------------

    test_df = test_df.copy()
    test_df["prob"] = probs

    patient_summary = (
        test_df
        .groupby(
            ["Patient_ID", "y"]
        )["prob"]
        .median()
        .reset_index()
    )

    patient_probs.extend(
        patient_summary["prob"].tolist()
    )

    patient_labels.extend(
        patient_summary["y"].tolist()
    )

    fold += 1

# ============================================================
# SEGMENT-LEVEL AUC
# ============================================================

segment_auc = roc_auc_score(
    all_true,
    all_probs
)

# ============================================================
# PATIENT-LEVEL AUC
# ============================================================

patient_auc = roc_auc_score(
    patient_labels,
    patient_probs
)

# ============================================================
# RESULTS
# ============================================================

print("\n" + "="*60)
print("ANALYSIS B RESULTS")
print("="*60)

print(f"Segment-Level LOSO AUC = {segment_auc:.4f}")
print(f"Patient-Level LOSO AUC = {patient_auc:.4f}")

# ============================================================
# FEATURE STABILITY
# ============================================================

stability_df = pd.DataFrame(
    feature_frequency.items(),
    columns=[
        "Feature",
        "Selection_Count"
    ]
)

stability_df["Selection_%"] = (
    100
    * stability_df["Selection_Count"]
    / df["Patient_ID"].nunique()
)

stability_df = stability_df.sort_values(
    "Selection_%",
    ascending=False
)

print("\nFeature Stability")
print(stability_df)

stability_df.to_csv(
    "/content/drive/MyDrive/AnalysisB_FeatureStability.csv",
    index=False
)


Fold 1
Selected: ['sampleEntropy', 'skewness', 'positiveToNegativeSampleRatio', 'approximateEntropy', 'renyiEntropy', 'kurtosis', 'spectralEntropy', 'fd_standardDeviation', 'singularValueDecompositionEntropy']

Fold 2
Selected: ['approximateEntropy', 'positiveToNegativeSampleRatio', 'spectralEntropy', 'renyiEntropy', 'skewness', 'maximum', 'higuchiFd', 'fd_standardDeviation', 'kurtosis']

Fold 3
Selected: ['sampleEntropy', 'skewness', 'standardDeviation', 'renyiEntropy', 'positiveToNegativeSampleRatio', 'spectralEntropy', 'maximum']

Fold 4
Selected: ['skewness', 'sampleEntropy', 'positiveToNegativeSampleRatio', 'higuchiFd', 'renyiEntropy', 'permutationEntropy', 'standardDeviation', 'fd_median', 'spectralEntropy']

Fold 5
Selected: ['positiveToNegativeSampleRatio', 'renyiEntropy', 'skewness', 'spectralEntropy', 'sampleEntropy', 'standardDeviation', 'kurtosis', 'permutationEntropy']

Fold 6
Selected: ['sampleEntropy', 'spectralEntropy', 'positiveToNegativeSampleRatio', 'skewness', 'kur

Analysis C

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
     "/content/drive/MyDrive/2min_12_Patients_35Features.csv"
)

# ============================================================
# PATIENT ID
# ============================================================

df["Patient_ID"] = (
    df["File_Name"]
    .astype(str)
    .str.extract(r'^(\d+)')
)

# Remove incomplete patient
df = df[df["Patient_ID"] != "98182"].copy()

# ============================================================
# LABELS
# ============================================================

df["y"] = df["label"].map({
    "before_HI": 0,
    "during_HI": 1
})

# ============================================================
# FEATURES
# ============================================================

exclude_cols = [
    "Sl. No:",
    "Segment_Number",
    "File_Name",
    "Patient_ID",
    "label",
    "y"
]

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
]

# ============================================================
# PATIENT BASELINE NORMALIZATION
# ============================================================

normalized_df = df.copy()

for patient in normalized_df["Patient_ID"].unique():

    patient_mask = (
        normalized_df["Patient_ID"] == patient
    )

    baseline_mask = (
        patient_mask
        &
        (normalized_df["label"] == "before_HI")
    )

    baseline = normalized_df.loc[
        baseline_mask,
        feature_cols
    ]

    baseline_mean = baseline.mean()

    baseline_std = baseline.std()

    baseline_std = baseline_std.replace(
        0,
        1
    )

    normalized_df.loc[
        patient_mask,
        feature_cols
    ] = (
        normalized_df.loc[
            patient_mask,
            feature_cols
        ]
        -
        baseline_mean
    ) / baseline_std

# ============================================================
# LOSO
# ============================================================

logo = LeaveOneGroupOut()

groups = normalized_df["Patient_ID"]

all_probs = []
all_true = []

patient_probs = []
patient_labels = []

feature_frequency = {}

# ============================================================
# LOSO LOOP
# ============================================================

fold = 1

for train_idx, test_idx in logo.split(
        normalized_df,
        normalized_df["y"],
        groups):

    print(f"\nFold {fold}")

    train_df = normalized_df.iloc[train_idx].copy()
    test_df = normalized_df.iloc[test_idx].copy()

    X_train_full = train_df[feature_cols]
    y_train = train_df["y"]

    # --------------------------------------------------------
    # MUTUAL INFORMATION
    # --------------------------------------------------------

    mi = mutual_info_classif(
        X_train_full,
        y_train,
        random_state=42
    )

    mi_df = pd.DataFrame({
        "Feature": feature_cols,
        "MI": mi
    })

    selected = (
        mi_df
        .sort_values(
            "MI",
            ascending=False
        )
        .head(10)
        ["Feature"]
        .tolist()
    )

    # --------------------------------------------------------
    # CORRELATION PRUNING
    # --------------------------------------------------------

    corr = (
        train_df[selected]
        .corr()
        .abs()
    )

    upper = corr.where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )

    remove = [
        col
        for col in upper.columns
        if any(upper[col] > 0.90)
    ]

    selected = [
        f for f in selected
        if f not in remove
    ]

    print("Selected:", selected)

    for f in selected:
        feature_frequency[f] = (
            feature_frequency.get(f, 0)
            + 1
        )

    X_train = train_df[selected]
    X_test = test_df[selected]

    y_train = train_df["y"]
    y_test = test_df["y"]

    # --------------------------------------------------------
    # RANDOM FOREST
    # --------------------------------------------------------

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    probs = model.predict_proba(
        X_test
    )[:, 1]

    all_probs.extend(probs)
    all_true.extend(y_test)

    # --------------------------------------------------------
    # PATIENT LEVEL
    # --------------------------------------------------------

    temp = test_df.copy()

    temp["prob"] = probs

    summary = (
        temp
        .groupby(
            ["Patient_ID", "y"]
        )["prob"]
        .median()
        .reset_index()
    )

    patient_probs.extend(
        summary["prob"]
    )

    patient_labels.extend(
        summary["y"]
    )

    fold += 1

# ============================================================
# RESULTS
# ============================================================

segment_auc = roc_auc_score(
    all_true,
    all_probs
)

patient_auc = roc_auc_score(
    patient_labels,
    patient_probs
)

print("\n" + "="*60)
print("ANALYSIS C RESULTS")
print("="*60)

print(
    f"Segment-Level LOSO AUC = "
    f"{segment_auc:.4f}"
)

print(
    f"Patient-Level LOSO AUC = "
    f"{patient_auc:.4f}"
)

# ============================================================
# FEATURE STABILITY
# ============================================================

stability_df = pd.DataFrame(
    feature_frequency.items(),
    columns=[
        "Feature",
        "Selection_Count"
    ]
)

stability_df["Selection_%"] = (
    100
    * stability_df["Selection_Count"]
    / normalized_df["Patient_ID"].nunique()
)

stability_df = stability_df.sort_values(
    "Selection_%",
    ascending=False
)

print("\nFeature Stability")
print(stability_df)


Fold 1
Selected: ['meanAbsoluteValue', 'sampleEntropy', 'higuchiFd', 'renyiEntropy', 'hjorthComplexity', 'positiveToNegativeSampleRatio']

Fold 2
Selected: ['fisherInfo', 'meanAbsoluteValue', 'higuchiFd', 'renyiEntropy', 'hjorthComplexity', 'permutationEntropy', 'approximateEntropy', 'positiveToNegativeSampleRatio']

Fold 3
Selected: ['meanAbsoluteValue', 'sampleEntropy', 'higuchiFd', 'fisherInfo', 'renyiEntropy', 'permutationEntropy', 'hjorthComplexity']

Fold 4
Selected: ['fisherInfo', 'permutationEntropy', 'higuchiFd', 'meanAbsoluteValue', 'sampleEntropy', 'positiveToNegativeSampleRatio', 'spectralEntropy']

Fold 5
Selected: ['meanAbsoluteValue', 'positiveToNegativeSampleRatio', 'sampleEntropy', 'fisherInfo', 'median', 'detrendedFluctuation', 'renyiEntropy', 'higuchiFd', 'kurtosis', 'permutationEntropy']

Fold 6
Selected: ['meanAbsoluteValue', 'sampleEntropy', 'higuchiFd', 'fisherInfo', 'positiveToNegativeSampleRatio', 'permutationEntropy', 'hjorthComplexity']

Fold 7
Selected: ['h

Analysis D:




Apply patient baseline normalization (Analysis C)
Run LOSO with:
Mutual Information feature selection
Mann–Whitney feature selection
Compute:
Segment-level AUC
Patient-level AUC
Per-fold AUCs
Feature stability
Verify the patient-level AUC calculation
Compare MI vs Mann–Whitney side-by-side

In [4]:
import pandas as pd
import numpy as np

from scipy.stats import mannwhitneyu
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ============================================================
# LOAD DATA
# ============================================================

DATA_PATH = "/content/drive/MyDrive/2min_12_Patients_35Features.csv"

df = pd.read_csv(DATA_PATH)

# ============================================================
# PATIENT ID
# ============================================================

df["Patient_ID"] = (
    df["File_Name"]
    .astype(str)
    .str.extract(r'^(\d+)')
)

# remove incomplete patient
df = df[df["Patient_ID"] != "98182"].copy()

# labels
df["y"] = df["label"].map({
    "before_HI": 0,
    "during_HI": 1
})

# ============================================================
# FEATURES
# ============================================================

exclude_cols = [
    "Sl. No:",
    "Segment_Number",
    "File_Name",
    "Patient_ID",
    "label",
    "y"
]

FEATURES = [
    c for c in df.columns
    if c not in exclude_cols
]

print("Patients:", df["Patient_ID"].nunique())
print("Features:", len(FEATURES))

# ============================================================
# BASELINE NORMALIZATION
# ============================================================

norm_df = df.copy()

for patient in norm_df["Patient_ID"].unique():

    patient_mask = (
        norm_df["Patient_ID"] == patient
    )

    baseline_mask = (
        patient_mask
        &
        (norm_df["label"] == "before_HI")
    )

    baseline = norm_df.loc[
        baseline_mask,
        FEATURES
    ]

    mu = baseline.mean()

    sigma = baseline.std()

    sigma = sigma.replace(0, 1)

    norm_df.loc[
        patient_mask,
        FEATURES
    ] = (
        norm_df.loc[
            patient_mask,
            FEATURES
        ]
        -
        mu
    ) / sigma

# ============================================================
# LOSO FUNCTION
# ============================================================

def run_loso(data, method="MI"):

    logo = LeaveOneGroupOut()

    groups = data["Patient_ID"]

    all_probs = []
    all_true = []

    patient_probs = []
    patient_labels = []

    feature_frequency = {}

    fold_results = []

    patient_summary_all = []

    print("\n")
    print("="*70)
    print(method)
    print("="*70)

    for fold,(train_idx,test_idx) in enumerate(
        logo.split(data,data["y"],groups),
        start=1
    ):

        train_df = data.iloc[train_idx].copy()
        test_df = data.iloc[test_idx].copy()

        held_out = test_df["Patient_ID"].iloc[0]

        X_train = train_df[FEATURES]
        y_train = train_df["y"]

        # ----------------------------------------------------
        # FEATURE SELECTION
        # ----------------------------------------------------

        if method == "MI":

            mi = mutual_info_classif(
                X_train,
                y_train,
                random_state=42
            )

            rank_df = pd.DataFrame({
                "Feature": FEATURES,
                "Score": mi
            })

            selected = (
                rank_df
                .sort_values(
                    "Score",
                    ascending=False
                )
                .head(10)
                ["Feature"]
                .tolist()
            )

        elif method == "MW":

            patient_train = (
                train_df
                .groupby(
                    ["Patient_ID","y"]
                )[FEATURES]
                .median()
                .reset_index()
            )

            scores = {}

            for f in FEATURES:

                before = patient_train[
                    patient_train["y"] == 0
                ][f]

                during = patient_train[
                    patient_train["y"] == 1
                ][f]

                try:
                    _, p = mannwhitneyu(
                        before,
                        during,
                        alternative="two-sided"
                    )
                except:
                    p = 1

                scores[f] = p

            selected = sorted(
                scores,
                key=scores.get
            )[:10]

        # ----------------------------------------------------
        # CORRELATION PRUNING
        # ----------------------------------------------------

        corr = (
            train_df[selected]
            .corr()
            .abs()
        )

        upper = corr.where(
            np.triu(
                np.ones(corr.shape),
                k=1
            ).astype(bool)
        )

        remove = [
            col
            for col in upper.columns
            if any(upper[col] > 0.90)
        ]

        selected = [
            f for f in selected
            if f not in remove
        ]

        print(
            f"Fold {fold} "
            f"({held_out}) "
            f"Features: {selected}"
        )

        # feature stability
        for f in selected:
            feature_frequency[f] = (
                feature_frequency.get(f,0)+1
            )

        # ----------------------------------------------------
        # RANDOM FOREST
        # ----------------------------------------------------

        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=5,
            random_state=42
        )

        model.fit(
            train_df[selected],
            y_train
        )

        probs = model.predict_proba(
            test_df[selected]
        )[:,1]

        y_test = test_df["y"]

        # segment level
        all_probs.extend(probs)
        all_true.extend(y_test)

        # fold auc
        fold_auc = roc_auc_score(
            y_test,
            probs
        )

        fold_results.append(
            [held_out, fold_auc]
        )

        # ----------------------------------------------------
        # PATIENT LEVEL
        # ----------------------------------------------------

        temp = test_df.copy()

        temp["prob"] = probs

        patient_summary = (
            temp
            .groupby(
                ["Patient_ID","y"]
            )["prob"]
            .median()
            .reset_index()
        )

        patient_summary_all.append(
            patient_summary
        )

        patient_probs.extend(
            patient_summary["prob"]
        )

        patient_labels.extend(
            patient_summary["y"]
        )

    # ========================================================
    # RESULTS
    # ========================================================

    segment_auc = roc_auc_score(
        all_true,
        all_probs
    )

    patient_auc = roc_auc_score(
        patient_labels,
        patient_probs
    )

    print("\n")
    print("="*70)
    print("RESULTS")
    print("="*70)

    print(
        f"Segment-Level AUC = "
        f"{segment_auc:.4f}"
    )

    print(
        f"Patient-Level AUC = "
        f"{patient_auc:.4f}"
    )

    # --------------------------------------------------------
    # FOLD SPREAD
    # --------------------------------------------------------

    fold_df = pd.DataFrame(
        fold_results,
        columns=[
            "Patient",
            "Fold_AUC"
        ]
    )

    print("\nPer-fold AUC")
    print(fold_df)

    print("\nFold statistics")
    print(
        fold_df["Fold_AUC"]
        .describe()
    )

    # --------------------------------------------------------
    # FEATURE STABILITY
    # --------------------------------------------------------

    stab = pd.DataFrame(
        feature_frequency.items(),
        columns=[
            "Feature",
            "Count"
        ]
    )

    stab["Percent"] = (
        100
        * stab["Count"]
        / data["Patient_ID"].nunique()
    )

    stab = stab.sort_values(
        "Percent",
        ascending=False
    )

    print("\nFeature Stability")
    print(stab)

    # --------------------------------------------------------
    # VERIFY PATIENT AUC
    # --------------------------------------------------------

    patient_summary_all = pd.concat(
        patient_summary_all,
        ignore_index=True
    )

    print("\nPatient Summary")
    print(
        patient_summary_all.head(20)
    )

    verify_auc = roc_auc_score(
        patient_summary_all["y"],
        patient_summary_all["prob"]
    )

    print(
        "\nVerified Patient AUC:",
        round(verify_auc,4)
    )

    return {
        "segment_auc": segment_auc,
        "patient_auc": patient_auc,
        "fold_df": fold_df,
        "stability": stab,
        "patient_summary": patient_summary_all
    }

# ============================================================
# RUN BOTH METHODS
# ============================================================

mi_results = run_loso(
    norm_df,
    method="MI"
)

mw_results = run_loso(
    norm_df,
    method="MW"
)

# ============================================================
# FINAL COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Method":[
        "Mutual Information",
        "Mann-Whitney"
    ],
    "Segment_AUC":[
        mi_results["segment_auc"],
        mw_results["segment_auc"]
    ],
    "Patient_AUC":[
        mi_results["patient_auc"],
        mw_results["patient_auc"]
    ]
})

print("\n")
print("="*70)
print("FINAL COMPARISON")
print("="*70)
print(comparison)

Patients: 11
Features: 35


MI
Fold 1 (1855) Features: ['meanAbsoluteValue', 'sampleEntropy', 'higuchiFd', 'renyiEntropy', 'hjorthComplexity', 'positiveToNegativeSampleRatio']
Fold 2 (26398) Features: ['fisherInfo', 'meanAbsoluteValue', 'higuchiFd', 'renyiEntropy', 'hjorthComplexity', 'permutationEntropy', 'approximateEntropy', 'positiveToNegativeSampleRatio']
Fold 3 (26502) Features: ['meanAbsoluteValue', 'sampleEntropy', 'higuchiFd', 'fisherInfo', 'renyiEntropy', 'permutationEntropy', 'hjorthComplexity']
Fold 4 (27245) Features: ['fisherInfo', 'permutationEntropy', 'higuchiFd', 'meanAbsoluteValue', 'sampleEntropy', 'positiveToNegativeSampleRatio', 'spectralEntropy']
Fold 5 (30071) Features: ['meanAbsoluteValue', 'positiveToNegativeSampleRatio', 'sampleEntropy', 'fisherInfo', 'median', 'detrendedFluctuation', 'renyiEntropy', 'higuchiFd', 'kurtosis', 'permutationEntropy']
Fold 6 (30851) Features: ['meanAbsoluteValue', 'sampleEntropy', 'higuchiFd', 'fisherInfo', 'positiveToNegativeSampl